<a href="https://colab.research.google.com/github/diegoRecinos/Proyecto_AN/blob/main/Proyecto_AN_Procesamiento_Imagenes_FFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Procesamiento de Imagenes con la FFT 2D

**Universidad Centroamericana Jose Simeon Canas**  
**Facultad de Ingenieria y Arquitectura - Departamento de Matematica**  
**Analisis Numerico - Ciclo 01-2026**

---

## Objetivo

Explorar la aplicacion de la **Transformada Discreta de Fourier en dos dimensiones (2D DFT / 2D FFT)** al procesamiento de imagenes digitales. Se estudiaran las siguientes tecnicas:

1. Representacion de una imagen en el dominio de frecuencias.
2. Filtrado paso-bajas (*Gaussian Smoothing*).
3. Filtrado paso-altas (deteccion de bordes).
4. Eliminacion de ruido periodico (filtro Notch).
5. Compresion de imagen mediante truncamiento de coeficientes.

> **Nota:** Todo el codigo usa `numpy.fft` como se indica en el proyecto.

---
## 1. Preliminares Matematicos

### 1.1 DFT en 1D

Dada una senal discreta $x[n]$ de longitud $N$:

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j2\pi kn/N}$$

### 1.2 DFT en 2D

Para una imagen $f[m,n]$ de tamano $M \times N$:

$$F[u,v] = \sum_{m=0}^{M-1}\sum_{n=0}^{N-1} f[m,n]\, e^{-j2\pi\left(\frac{um}{M}+\frac{vn}{N}\right)}$$

La FFT reduce la complejidad de $O(M^2 N^2)$ a $O(MN\log(MN))$ aplicando FFT 1D por filas y columnas.

### 1.3 Espectro de magnitud

Como $F[u,v] \in \mathbb{C}$, se visualiza con escala logaritmica:

$$S[u,v] = \log(1 + |F[u,v]|)$$

---
## 2. Importacion de Librerias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage import data
from skimage.util import img_as_float

plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'font.family': 'serif'})
print('Librerias cargadas.')
print('NumPy:', np.__version__)

---
## 3. Funciones Auxiliares

In [ ]:
def compute_fft2(imagen):
    """Calcula FFT 2D centrada y su espectro log."""
    F = np.fft.fft2(imagen)
    F_shifted = np.fft.fftshift(F)
    mag_log = np.log1p(np.abs(F_shifted))
    return F, F_shifted, mag_log


def mostrar_par(img1, img2, t1, t2, cmap='gray'):
    """Muestra dos imagenes lado a lado."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img1, cmap=cmap); axes[0].set_title(t1); axes[0].axis('off')
    axes[1].imshow(img2, cmap=cmap); axes[1].set_title(t2); axes[1].axis('off')
    plt.tight_layout(); plt.show()


def filtro_gaussiano_fft(imagen, sigma):
    """Filtro Gaussiano paso-bajas via FFT."""
    M, N = imagen.shape
    F = np.fft.fft2(imagen)
    u = np.fft.fftfreq(M, d=1.0/M)
    v = np.fft.fftfreq(N, d=1.0/N)
    V, U = np.meshgrid(v, u)
    H = np.exp(-(U**2 + V**2) / (2 * sigma**2))
    img_filtrada = np.real(np.fft.ifft2(F * H))
    return img_filtrada, H


print('Funciones definidas correctamente.')

---
## 4. Imagen de Prueba y Espectro de Frecuencias

In [ ]:
# Imagen de prueba: 'camera' de scikit-image (512x512)
imagen = img_as_float(data.camera())

F, F_shifted, mag_log = compute_fft2(imagen)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(imagen, cmap='gray')
axes[0].set_title('Imagen original - dominio espacial')
axes[0].axis('off')
axes[1].imshow(mag_log, cmap='inferno')
axes[1].set_title('Espectro de magnitud (log scale)')
axes[1].axis('off')
axes[2].imshow(np.angle(F_shifted), cmap='hsv')
axes[2].set_title('Espectro de fase')
axes[2].axis('off')
plt.suptitle('Dominio espacial vs. dominio de frecuencias', fontsize=14)
plt.tight_layout(); plt.show()

print('Tamano imagen:', imagen.shape)
print('Tipo F:', F.dtype)

### Interpretacion

- El **centro** del espectro (tras `fftshift`) corresponde a la frecuencia DC: brillo promedio de la imagen.
- Frecuencias **bajas** (cerca del centro): variaciones lentas, fondos y gradientes.
- Frecuencias **altas** (lejos del centro): bordes, texturas y ruido.
- El espectro es **simetrico hermiticamente** porque la imagen es real: $F[-u,-v] = F^*[u,v]$.

---
## 5. Filtrado Paso-Bajas: Suavizado Gaussiano

El filtro Gaussiano en frecuencias:

$$H[u,v] = e^{-\dfrac{u^2+v^2}{2\sigma^2}}$$

Es equivalente (por el Teorema de Convolucion) a convolucionar con una Gaussiana en el dominio espacial:

$$\mathcal{F}\{f * g\} = F \cdot G$$

In [ ]:
sigmas = [5, 20, 60]
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

axes[0, 0].imshow(imagen, cmap='gray'); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
_, _, m0 = compute_fft2(imagen)
axes[1, 0].imshow(m0, cmap='inferno'); axes[1, 0].set_title('Espectro original'); axes[1, 0].axis('off')

for i, sigma in enumerate(sigmas, start=1):
    img_f, _ = filtro_gaussiano_fft(imagen, sigma)
    _, _, mag_f = compute_fft2(img_f)
    axes[0, i].imshow(np.clip(img_f, 0, 1), cmap='gray')
    axes[0, i].set_title('Gaussiano sigma=' + str(sigma))
    axes[0, i].axis('off')
    axes[1, i].imshow(mag_f, cmap='inferno')
    axes[1, i].set_title('Espectro sigma=' + str(sigma))
    axes[1, i].axis('off')

plt.suptitle('Filtrado paso-bajas Gaussiano', fontsize=14)
plt.tight_layout(); plt.show()

---
## 6. Filtrado Paso-Altas: Deteccion de Bordes

El complemento del Gaussiano elimina bajas frecuencias y conserva los bordes:

$$H_{\text{altas}}[u,v] = 1 - e^{-\frac{u^2+v^2}{2\sigma^2}}$$

In [ ]:
def filtro_paso_altas_fft(imagen, sigma):
    M, N = imagen.shape
    F = np.fft.fft2(imagen)
    u = np.fft.fftfreq(M, d=1.0/M)
    v = np.fft.fftfreq(N, d=1.0/N)
    V, U = np.meshgrid(v, u)
    H_altas = 1 - np.exp(-(U**2 + V**2) / (2 * sigma**2))
    return np.real(np.fft.ifft2(F * H_altas))


bordes = filtro_paso_altas_fft(imagen, sigma=20)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(imagen, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(bordes, cmap='gray'); axes[1].set_title('Bordes (paso-altas)'); axes[1].axis('off')
_, _, mb = compute_fft2(bordes)
axes[2].imshow(mb, cmap='inferno'); axes[2].set_title('Espectro de bordes'); axes[2].axis('off')
plt.suptitle('Filtrado paso-altas - Deteccion de bordes', fontsize=14)
plt.tight_layout(); plt.show()

---
## 7. Eliminacion de Ruido Periodico (Filtro Notch)

El ruido periodico aparece en el espectro como **picos brillantes** en posiciones $(u_0, v_0)$.
Se eliminan con una mascara que anula esas frecuencias especificas:

$$H_{\text{notch}}[u,v] = \begin{cases} 0 & \text{pico de ruido}\\ 1 & \text{otro caso} \end{cases}$$

In [ ]:
# Agregar ruido sinusoidal periodico
M, N = imagen.shape
freq_ruido = 30
x, y = np.arange(N), np.arange(M)
X, Y = np.meshgrid(x, y)
amplitud = 0.15
ruido = amplitud * np.sin(2*np.pi*freq_ruido*X/N) + amplitud * np.sin(2*np.pi*freq_ruido*Y/M)
imagen_ruidosa = np.clip(imagen + ruido, 0, 1)
mostrar_par(imagen, imagen_ruidosa, 'Original', 'Con ruido periodico')

In [ ]:
def eliminar_ruido_periodico(imagen, freq_ruido, radio_notch=4):
    M, N = imagen.shape
    F = np.fft.fft2(imagen)
    F_shifted = np.fft.fftshift(F)
    mascara = np.ones((M, N), dtype=float)
    cm, cn = M // 2, N // 2
    picos = [(cm, cn+freq_ruido), (cm, cn-freq_ruido), (cm+freq_ruido, cn), (cm-freq_ruido, cn)]
    ui, vi = np.ogrid[:M, :N]
    for (pu, pv) in picos:
        dist = np.sqrt((ui-pu)**2 + (vi-pv)**2)
        mascara[dist <= radio_notch] = 0
    img_limpia = np.real(np.fft.ifft2(np.fft.ifftshift(F_shifted * mascara)))
    return img_limpia, mascara


img_limpia, mascara = eliminar_ruido_periodico(imagen_ruidosa, freq_ruido)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(imagen_ruidosa, cmap='gray'); axes[0].set_title('Con ruido'); axes[0].axis('off')
axes[1].imshow(mascara, cmap='gray'); axes[1].set_title('Mascara notch'); axes[1].axis('off')
axes[2].imshow(np.clip(img_limpia, 0, 1), cmap='gray'); axes[2].set_title('Restaurada'); axes[2].axis('off')
plt.suptitle('Filtro Notch - Eliminacion de ruido periodico', fontsize=14)
plt.tight_layout(); plt.show()

mse_r = np.mean((imagen - imagen_ruidosa)**2)
mse_l = np.mean((imagen - np.clip(img_limpia, 0, 1))**2)
print('MSE (ruidosa vs original)   :', round(mse_r, 6))
print('MSE (restaurada vs original):', round(mse_l, 6))
print('Mejora relativa             :', round((1 - mse_l/mse_r)*100, 1), '%')

---
## 8. Compresion de Imagen mediante FFT

Se retienen solo los $k$ coeficientes con mayor energia (Teorema de Parseval):

$$\sum_{m,n} |f[m,n]|^2 = \frac{1}{MN}\sum_{u,v} |F[u,v]|^2$$

La calidad se mide con el **PSNR**:

$$\text{PSNR} = 10\log_{10}\!\left(\frac{1}{\text{MSE}}\right) \quad \text{(para imagenes en [0,1])}$$

In [ ]:
def comprimir_fft(imagen, porcentaje_coef):
    F = np.fft.fft2(imagen)
    umbral = np.percentile(np.abs(F).ravel(), 100 - porcentaje_coef)
    F_c = F.copy()
    F_c[np.abs(F_c) < umbral] = 0
    return np.clip(np.real(np.fft.ifft2(F_c)), 0, 1)


porcentajes = [10, 5, 1, 0.2]
fig, axes = plt.subplots(2, 5, figsize=(20, 9))
axes[0, 0].imshow(imagen, cmap='gray'); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
_, _, m0 = compute_fft2(imagen)
axes[1, 0].imshow(m0, cmap='inferno'); axes[1, 0].set_title('Espectro 100%'); axes[1, 0].axis('off')

for i, pct in enumerate(porcentajes, start=1):
    img_c = comprimir_fft(imagen, pct)
    mse = np.mean((imagen - img_c)**2)
    psnr = 10 * np.log10(1 / mse) if mse > 0 else 60.0
    axes[0, i].imshow(img_c, cmap='gray')
    axes[0, i].set_title(str(pct) + '% coef.  PSNR=' + str(round(psnr, 1)) + ' dB')
    axes[0, i].axis('off')
    _, _, mc = compute_fft2(img_c)
    axes[1, i].imshow(mc, cmap='inferno')
    axes[1, i].set_title('Espectro ' + str(pct) + '%')
    axes[1, i].axis('off')

plt.suptitle('Compresion FFT - truncamiento de coeficientes', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Curva PSNR vs porcentaje de coeficientes
pcts = np.logspace(-1, 2, 40)
psnrs = []
for pct in pcts:
    img_c = comprimir_fft(imagen, min(float(pct), 100.0))
    mse = np.mean((imagen - img_c)**2)
    psnrs.append(10 * np.log10(1 / mse) if mse > 0 else 60.0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(pcts, psnrs, 'b-o', markersize=4, linewidth=1.8)
ax.axhline(30, color='r', linestyle='--', label='Umbral calidad (30 dB)')
ax.set_xlabel('Porcentaje de coeficientes FFT conservados (%)')
ax.set_ylabel('PSNR (dB)')
ax.set_title('Calidad de reconstruccion vs. tasa de compresion')
ax.legend(); ax.grid(True, which='both', alpha=0.4)
plt.tight_layout(); plt.show()

---
## 9. Verificacion del Teorema de Convolucion

Por el **Teorema de la Convolucion**, filtrar en frecuencias equivale a convolucionar en espacio. Comparamos nuestra implementacion FFT con `scipy.ndimage.gaussian_filter` (referencia).

In [ ]:
sigma_test = 3.0
img_scipy = gaussian_filter(imagen, sigma=sigma_test)
img_fft_v, _ = filtro_gaussiano_fft(imagen, sigma=sigma_test)
img_fft_v = np.clip(img_fft_v, 0, 1)
diferencia = np.abs(img_scipy - img_fft_v)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_scipy, cmap='gray'); axes[0].set_title('Gaussiano SciPy'); axes[0].axis('off')
axes[1].imshow(img_fft_v, cmap='gray'); axes[1].set_title('Gaussiano FFT (NumPy)'); axes[1].axis('off')
im = axes[2].imshow(diferencia, cmap='hot')
axes[2].set_title('Diferencia abs (max=' + '{:.2e}'.format(diferencia.max()) + ')')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.suptitle('Verificacion del Teorema de Convolucion', fontsize=14)
plt.tight_layout(); plt.show()

print('Diferencia maxima   :', '{:.2e}'.format(diferencia.max()))
print('Diferencia promedio :', '{:.2e}'.format(diferencia.mean()))
print('-> Resultados equivalentes (diferencias por efectos de borde).')

---
## 10. Conclusiones

| Tecnica | Descripcion | Aplicacion |
|---|---|---|
| FFT 2D | Transforma al dominio de frecuencias | Base de todos los filtros |
| Paso-bajas Gaussiano | Atenua frecuencias altas | Desenfoque, eliminacion de ruido |
| Paso-altas | Atenua frecuencias bajas | Deteccion de bordes |
| Filtro Notch | Elimina frecuencias especificas | Ruido periodico |
| Compresion FFT | Trunca coeficientes de baja energia | Base conceptual de JPEG |

1. La FFT 2D procesa imagenes en $O(MN\log MN)$, mucho mas eficiente que la DFT directa.
2. El Teorema de Convolucion conecta el filtrado en frecuencias con la convolucion espacial.
3. Las frecuencias bajas dominan la apariencia; las altas codifican bordes y textura.
4. La compresion FFT muestra que pocas frecuencias concentran la mayor parte de la energia.
5. El filtro Notch elimina ruido periodico de forma selectiva y eficaz.

## Referencias

1. Tan, X. (2020). *Fast Fourier Transform*. University of Connecticut.
2. Kulkarni, S. R. *Frequency Domain and Fourier Transforms*. Princeton University.
3. Python Numerical Methods at Berkeley. *Discrete Fourier Transform*.
4. Gonzalez, R. C., & Woods, R. E. (2018). *Digital Image Processing* (4th ed.). Pearson.
5. NumPy Documentation. `numpy.fft` module.